**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

# Local Features Studio — Computer Vision 2026

**Lezione 4 · Laurea Magistrale in Ingegneria Informatica, secondo anno**  
**Durata:** 3 ore · **Modalità:** coppie, confronto finale collettivo

## Domanda guida

> Quando una feature è ripetibile, quando un match è credibile e come possiamo dimostrarlo?

Il notebook costruisce una pipeline osservabile:

**immagine → filtraggio e gradienti → detector → keypoint → descrittore → matching → verifica geometrica**.

Non cerchiamo “parametri che funzionano”. Formuliamo una previsione, cambiamo una sola causa, misuriamo l'effetto e diagnostichiamo il primo stadio che fallisce.

Le misure dipendono dalle versioni delle librerie; il run di riferimento usa OpenCV 4.12.0 e NumPy 2.2.6.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

## Percorso e consegne

| Esperimento | Evidenza richiesta |
| --- | --- |
| ambiente, scena e gradienti | figura dello sweep di $\sigma$ |
| Harris e Shi–Tomasi | tabella densità/copertura |
| ripetibilità controllata | curva $\rho(\varepsilon)$ e heatmap |
| confronto tra detector | tabella a parità di budget |
| descrittori e ratio test | raw match e ratio sweep |
| RANSAC ed errore | inlier/outlier e soglia geometrica |
| failure clinic ed discussione conclusiva | diagnosi in cinque righe |

**Materiali da conservare:** notebook eseguito, quattro figure scelte, due tabelle, diagnosi finale e una coppia di immagini reali. Il risultato numerico senza protocollo non è sufficiente.

## Lab 0 — Ambiente riproducibile

Il notebook richiede soltanto NumPy, Pandas, Matplotlib e OpenCV. In Google Colab queste librerie sono normalmente disponibili. In un ambiente locale, se un import fallisce, installare i pacchetti elencati in [`requirements_lezione04_revisionata.txt`](requirements_lezione04_revisionata.txt) e riavviare il kernel.

Il seed rende ripetibile il campionamento di RANSAC; non rende deterministiche eventuali differenze tra versioni o implementazioni.

In [ ]:
import sys
import time
from pathlib import Path

import cv2 as cv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260924
np.random.seed(SEED)
cv.setRNGSeed(SEED)
cv.setNumThreads(1)

plt.rcParams.update({
    "figure.dpi": 105,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})

print("Python:", sys.version.split()[0])
print("OpenCV:", cv.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
assert int(cv.__version__.split(".")[0]) >= 4, "È richiesto OpenCV 4.x"


## Lab 1 — Una scena controllata

La tavola contiene strutture volutamente diverse:

- scacchiera: molti corner ma forte ripetizione;
- simbolo circolare: orientazioni curve;
- testo e codice `A7`: strutture distintive;
- linee oblique: bordi forti ma pochi corner ben localizzati.

Poiché generiamo noi la trasformazione, conosciamo la geometria vera $H_{gt}$ e possiamo distinguere una corrispondenza corretta da un semplice consenso di RANSAC.

In [ ]:
CV_INK = (24, 33, 47)
CV_CYAN = (64, 197, 185)
CV_CORAL = (240, 80, 84)
CV_VIOLET = (87, 73, 183)


def rgb(bgr):
    # OpenCV BGR -> Matplotlib RGB.
    return cv.cvtColor(bgr, cv.COLOR_BGR2RGB)


def gray_of(image):
    return cv.cvtColor(image, cv.COLOR_BGR2GRAY) if image.ndim == 3 else image


def make_test_board(width=960, height=640):
    img = np.full((height, width, 3), (238, 234, 224), np.uint8)
    cv.rectangle(img, (0, 0), (width - 1, height - 1), CV_INK, 20)
    cv.putText(img, "LOCAL FEATURES", (62, 78), cv.FONT_HERSHEY_SIMPLEX,
               1.55, (25, 40, 55), 4, cv.LINE_AA)
    sq, ox, oy = 42, 66, 170
    for row in range(7):
        for col in range(9):
            color = (35, 44, 58) if (row + col) % 2 == 0 else (246, 192, 62)
            cv.rectangle(img, (ox + col * sq, oy + row * sq),
                         (ox + (col + 1) * sq, oy + (row + 1) * sq), color, -1)
    cv.circle(img, (640, 252), 105, CV_CYAN, -1, cv.LINE_AA)
    cv.circle(img, (640, 252), 64, (25, 40, 55), 8, cv.LINE_AA)
    cv.line(img, (588, 200), (690, 304), CV_CORAL, 16, cv.LINE_AA)
    cv.line(img, (588, 304), (690, 200), CV_CORAL, 16, cv.LINE_AA)
    cv.putText(img, "A7", (754, 242), cv.FONT_HERSHEY_DUPLEX,
               2.05, CV_VIOLET, 5, cv.LINE_AA)
    for x in range(520, 890, 24):
        cv.line(img, (x, 410), (x + 50, 550), (42, 55, 69), 3, cv.LINE_AA)
    cv.rectangle(img, (505, 394), (900, 566), CV_CORAL, 5, cv.LINE_AA)
    return img


def warp_test(image, angle=28, scale=0.82, tx=72, ty=22,
              gain=0.78, bias=34, blur=0.8, border=(30, 38, 51)):
    # Trasformazione geometrica nota seguita da cambiamento fotometrico e blur.
    height, width = image.shape[:2]
    A = cv.getRotationMatrix2D((width / 2, height / 2), angle, scale)
    A[:, 2] += (tx, ty)
    transformed = cv.warpAffine(image, A, (width, height),
                                flags=cv.INTER_LINEAR, borderValue=border)
    if blur > 0:
        transformed = cv.GaussianBlur(transformed, (0, 0), blur)
    transformed = cv.convertScaleAbs(transformed, alpha=gain, beta=bias)
    H = np.vstack([A, [0.0, 0.0, 1.0]])
    return transformed, H


img1 = make_test_board()
gray1 = gray_of(img1)
img2, H_gt = warp_test(img1)
gray2 = gray_of(img2)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for ax, image, title in zip(axes, [img1, img2],
                            ["Immagine di riferimento", "Trasformazione controllata"]):
    ax.imshow(rgb(image)); ax.set_title(title); ax.axis("off")
plt.tight_layout()


### Esperimento 1A — Smoothing e gradienti

Variare una sola quantità, $\sigma$, mantenendo fisso tutto il resto. Misuriamo:

- il percentile 98 del modulo del gradiente, come indicatore della risposta forte;
- il numero di corner Shi–Tomasi rilevati con lo stesso budget massimo;
- ciò che scompare prima nelle tre regioni della tavola.

**Previsione prima di eseguire:** aumentando $\sigma$, quali strutture perderanno risposta per prime?

Le mappe del gradiente usano una scala cromatica comune, da zero al massimo del gradiente dell’immagine non filtrata. La luminosità è quindi confrontabile fra i pannelli.

In [ ]:
sigmas = [0.0, 0.8, 1.4, 2.4, 4.0]
gradient_rows = []
reference_gx = cv.Sobel(gray1, cv.CV_32F, 1, 0, ksize=3)
reference_gy = cv.Sobel(gray1, cv.CV_32F, 0, 1, ksize=3)
gradient_vmax = float(cv.magnitude(reference_gx, reference_gy).max())

fig, axes = plt.subplots(len(sigmas), 2, figsize=(12, 15))
for row, sigma in enumerate(sigmas):
    smooth = gray1 if sigma == 0 else cv.GaussianBlur(gray1, (0, 0), sigma)
    Ix = cv.Sobel(smooth, cv.CV_32F, 1, 0, ksize=3)
    Iy = cv.Sobel(smooth, cv.CV_32F, 0, 1, ksize=3)
    magnitude = cv.magnitude(Ix, Iy)
    corners = cv.goodFeaturesToTrack(smooth, maxCorners=800, qualityLevel=0.01,
                                     minDistance=6, blockSize=5)
    count = 0 if corners is None else len(corners)
    gradient_rows.append({
        "sigma": sigma,
        "p98_gradiente": float(np.percentile(magnitude, 98)),
        "corner": count,
    })
    axes[row, 0].imshow(smooth, cmap="gray", vmin=0, vmax=255)
    axes[row, 0].set_title(f"σ={sigma:g} — immagine filtrata")
    axes[row, 1].imshow(magnitude, cmap="magma", vmin=0, vmax=gradient_vmax)
    axes[row, 1].set_title(f"|∇I| — p98={gradient_rows[-1]['p98_gradiente']:.1f}")
    for ax in axes[row]: ax.axis("off")
plt.tight_layout()

gradient_df = pd.DataFrame(gradient_rows)
display(gradient_df.round(2))


**Risposta dello studente.** Confrontare scacchiera, testo e linee oblique. Quale evidenza sostiene la risposta?

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

## Lab 2 — Harris, Shi–Tomasi e selezione dei punti

La mappa Harris non è ancora un insieme di keypoint. Per ottenere punti sparsi servono soglia, massimi locali e distanza minima. `goodFeaturesToTrack` permette di osservare direttamente l'effetto di `qualityLevel` e `minDistance`.

In [ ]:
harris = cv.cornerHarris(np.float32(gray1), blockSize=5, ksize=3, k=0.04)
harris_vis = cv.dilate(harris, None)

shi = cv.goodFeaturesToTrack(gray1, maxCorners=220, qualityLevel=0.012,
                             minDistance=10, blockSize=5,
                             useHarrisDetector=False)
shi_points = np.empty((0, 2), np.float32) if shi is None else shi.reshape(-1, 2)

vis = img1.copy()
for x, y in shi_points:
    cv.circle(vis, (round(x), round(y)), 5, (79, 83, 255), 2, cv.LINE_AA)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(harris_vis, cmap="turbo")
axes[0].set_title("Harris response: positiva / negativa")
axes[1].imshow(rgb(vis))
axes[1].set_title(f"Shi–Tomasi: {len(shi_points)} punti sparsi")
for ax in axes: ax.axis("off")
plt.tight_layout()


### Esperimento 2A — Densità, ridondanza e copertura

La copertura divide l'immagine in una griglia $6\times10$ e calcola la frazione di celle che contiene almeno un punto. Non misura la qualità del singolo corner: misura quanto l'evidenza è distribuita.

In [ ]:
def grid_coverage(points, shape, rows=6, cols=10):
    if len(points) == 0:
        return 0.0
    height, width = shape[:2]
    col = np.clip((points[:, 0] / width * cols).astype(int), 0, cols - 1)
    row = np.clip((points[:, 1] / height * rows).astype(int), 0, rows - 1)
    return len(set(zip(row, col))) / (rows * cols)


def median_nearest_neighbor(points):
    points = np.asarray(points, dtype=float).reshape(-1, 2)
    if len(points) < 2:
        return np.nan
    distances = np.linalg.norm(points[:, None, :] - points[None, :, :], axis=2)
    np.fill_diagonal(distances, np.inf)
    return float(np.median(distances.min(axis=1)))


parameter_sets = [
    (0.004, 5), (0.004, 15),
    (0.012, 5), (0.012, 15),
    (0.030, 5), (0.030, 15),
]
selection_rows = []
selection_images = []
for quality, distance in parameter_sets:
    detected = cv.goodFeaturesToTrack(
        gray1, maxCorners=300, qualityLevel=quality,
        minDistance=distance, blockSize=5, useHarrisDetector=False)
    points = np.empty((0, 2), np.float32) if detected is None else detected.reshape(-1, 2)
    selection_rows.append({
        "qualityLevel": quality,
        "minDistance": distance,
        "punti": len(points),
        "copertura": grid_coverage(points, img1.shape),
        "distanza_NN_mediana": median_nearest_neighbor(points),
    })
    canvas = img1.copy()
    for x, y in points:
        cv.circle(canvas, (round(x), round(y)), 4, CV_CORAL, 2, cv.LINE_AA)
    selection_images.append(canvas)

selection_df = pd.DataFrame(selection_rows)
display(selection_df.round(3))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, canvas, row in zip(axes.ravel(), selection_images, selection_rows):
    ax.imshow(rgb(canvas))
    ax.set_title(f"q={row['qualityLevel']:.3f}, d={row['minDistance']}\n"
                 f"N={row['punti']}, coverage={row['copertura']:.2f}")
    ax.axis("off")
plt.tight_layout()


**Risposta dello studente.** Quale parametro agisce soprattutto sulla qualità della risposta e quale sulla ridondanza spaziale? Citare una riga della tabella.

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

## Lab 3 — Misurare la ripetibilità

Un punto $\mathbf p$ rilevato in $I_1$ è ripetuto se, dopo la proiezione con $H_{gt}$, esiste un punto di $I_2$ entro $\varepsilon$ pixel. Sono comuni due convenzioni:

- **diretta:** ogni punto visibile cerca il proprio vicino; lo stesso target può spiegare più detections;
- **uno-a-uno:** ogni detection target può essere usata una sola volta.

Le figure di riferimento e gli sweep riportano la convenzione uno-a-uno; il valore diretto è calcolato separatamente per confronto. La funzione seguente:

1. considera soltanto i punti proiettati nel supporto visibile;
2. impone associazioni uno-a-uno con una selezione greedy sulle distanze;
3. restituisce anche gli errori di localizzazione, non soltanto il rapporto.

**Convenzione di visibilità.** Si controlla il centro proiettato nel rettangolo dell’immagine target. Non si escludono automaticamente patch parzialmente fuori immagine o regioni occluse. Il matching uno-a-uno è greedy sulle distanze e non risolve un problema globale di assegnamento.

In [ ]:
def project_points(points, H):
    points = np.asarray(points, dtype=np.float64).reshape(-1, 2)
    if len(points) == 0:
        return np.empty((0, 2), np.float64)
    homogeneous = np.c_[points, np.ones(len(points))]
    projected = (H @ homogeneous.T).T
    return projected[:, :2] / projected[:, 2:3]


def repeatability(points1, points2, H, target_shape, epsilon=5.0):
    points1 = np.asarray(points1, dtype=np.float64).reshape(-1, 2)
    points2 = np.asarray(points2, dtype=np.float64).reshape(-1, 2)
    projected_all = project_points(points1, H)
    height, width = target_shape[:2]
    visible_mask = (
        (projected_all[:, 0] >= 0) & (projected_all[:, 0] < width) &
        (projected_all[:, 1] >= 0) & (projected_all[:, 1] < height)
    )
    source_indices = np.flatnonzero(visible_mask)
    projected = projected_all[visible_mask]
    if len(projected) == 0 or len(points2) == 0:
        return {"rho": 0.0, "repeated": 0, "visible": len(projected),
                "errors": np.array([]), "pairs": [], "projected": projected}

    distances = np.linalg.norm(projected[:, None, :] - points2[None, :, :], axis=2)
    candidates = np.argwhere(distances <= epsilon)
    candidates = sorted(candidates, key=lambda ij: distances[ij[0], ij[1]])
    used_source, used_target, pairs, errors = set(), set(), [], []
    for i, j in candidates:
        i, j = int(i), int(j)
        if i in used_source or j in used_target:
            continue
        used_source.add(i); used_target.add(j)
        pairs.append((int(source_indices[i]), j))
        errors.append(float(distances[i, j]))

    repeated = len(pairs)
    return {
        "rho": repeated / len(projected),
        "repeated": repeated,
        "visible": len(projected),
        "errors": np.asarray(errors),
        "pairs": pairs,
        "projected": projected,
    }


def directed_repeatability(points1, points2, H, target_shape, epsilon=5.0):
    projected = project_points(points1, H)
    height, width = target_shape[:2]
    visible = projected[(projected[:, 0] >= 0) & (projected[:, 0] < width) &
                        (projected[:, 1] >= 0) & (projected[:, 1] < height)]
    if len(visible) == 0 or len(points2) == 0:
        return {"rho": 0.0, "repeated": 0, "visible": len(visible)}
    nearest = np.linalg.norm(visible[:, None, :] - np.asarray(points2)[None, :, :], axis=2).min(axis=1)
    repeated = int(np.sum(nearest <= epsilon))
    return {"rho": repeated / len(visible), "repeated": repeated, "visible": len(visible)}


def detect_shi(gray, budget=220, quality=0.012, min_distance=10):
    found = cv.goodFeaturesToTrack(gray, maxCorners=budget, qualityLevel=quality,
                                   minDistance=min_distance, blockSize=5,
                                   useHarrisDetector=False)
    return np.empty((0, 2), np.float32) if found is None else found.reshape(-1, 2)


p1 = detect_shi(gray1)
p2 = detect_shi(gray2)
rep5 = repeatability(p1, p2, H_gt, img2.shape, epsilon=5)
directed5 = directed_repeatability(p1, p2, H_gt, img2.shape, epsilon=5)
print(f"Ripetibilità diretta @5 px = {directed5['rho']:.3f} "
      f"({directed5['repeated']}/{directed5['visible']})")
print(f"Ripetibilità uno-a-uno @5 px = {rep5['rho']:.3f} "
      f"({rep5['repeated']}/{rep5['visible']})")
print("Errore mediano dei ripetuti:",
      f"{np.median(rep5['errors']):.2f} px" if len(rep5['errors']) else "n.d.")


### Esperimento 3A — La curva $\rho(\varepsilon)$

La stessa ripetibilità finale può nascondere localizzazioni molto diverse. Eseguiamo lo sweep della tolleranza senza cambiare detector.

In [ ]:
epsilons = np.arange(1, 13)
epsilon_rows = []
for epsilon in epsilons:
    result = repeatability(p1, p2, H_gt, img2.shape, epsilon=float(epsilon))
    epsilon_rows.append({
        "epsilon_px": epsilon,
        "rho": result["rho"],
        "ripetuti": result["repeated"],
        "visibili": result["visible"],
    })
epsilon_df = pd.DataFrame(epsilon_rows)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(epsilon_df["epsilon_px"], epsilon_df["rho"], "o-", color="#008B9A")
ax.set(xlabel="tolleranza ε [pixel]", ylabel="ripetibilità ρ",
       ylim=(0, 1.03), title="Localizzazione: non basta un solo valore di ε")
ax.grid(alpha=0.25)
plt.show()
display(epsilon_df.round(3))


### Esperimento 3B — Curva di robustezza a rotazione e scala

Manteniamo fissi detector, budget, blur e variazione fotometrica. Per ogni cella della heatmap cambia soltanto la coppia `(scala, angolo)`.

In [ ]:
angles = [0, 15, 30, 45, 60]
scales = [1.0, 0.8, 0.6]
robustness_rows = []

for scale in scales:
    for angle in angles:
        transformed, H = warp_test(img1, angle=angle, scale=scale)
        detected = detect_shi(gray_of(transformed))
        result = repeatability(p1, detected, H, transformed.shape, epsilon=5)
        robustness_rows.append({
            "scala": scale,
            "angolo": angle,
            "punti_target": len(detected),
            "visibili": result["visible"],
            "ripetuti": result["repeated"],
            "rho": result["rho"],
        })

robustness_df = pd.DataFrame(robustness_rows)
pivot = robustness_df.pivot(index="scala", columns="angolo", values="rho")
display(robustness_df.round(3))

fig, ax = plt.subplots(figsize=(8.5, 3.8))
heat = ax.imshow(pivot.values, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_xlabel("rotazione [gradi]"); ax.set_ylabel("scala")
ax.set_title("Ripetibilità Shi–Tomasi @5 px")
for row in range(pivot.shape[0]):
    for col in range(pivot.shape[1]):
        value = pivot.iloc[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center",
                color="white" if value < 0.65 else "black")
fig.colorbar(heat, ax=ax, label="ρ")
plt.tight_layout()


**Risposta dello studente.** Dove diminuisce la robustezza, e di quanto? La causa più plausibile è rotazione, scala, supporto visibile o soglia?

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

> ### Checkpoint
>
> Prima della pausa confrontare con un altro gruppo:
>
> 1. stesso valore di $\rho$ significa stessa accuratezza di localizzazione?
> 2. perché i punti fuori dal supporto non sono falsi negativi?
> 3. quale parametro avete mantenuto fisso per rendere equo il confronto?

## Lab 4 — Detector a parità di budget

Confrontiamo Shi–Tomasi, FAST, ORB e, se disponibile, SIFT. Ogni funzione restituisce soltanto coordinate; scala e orientazione verranno analizzate nel laboratorio successivo. Il budget massimo è 220 punti.

**Nota:** il tempo misurato nel notebook è indicativo, non un benchmark rigoroso. Va usato soltanto per riconoscere differenze macroscopiche nello stesso ambiente.

In [ ]:
def keypoints_to_points(keypoints, budget=None):
    ordered = sorted(keypoints, key=lambda k: k.response, reverse=True)
    if budget is not None:
        ordered = ordered[:budget]
    return np.float32([k.pt for k in ordered]).reshape(-1, 2) if ordered else np.empty((0, 2), np.float32)


def detect_fast(gray, budget=220):
    detector = cv.FastFeatureDetector_create(threshold=18, nonmaxSuppression=True)
    return keypoints_to_points(detector.detect(gray, None), budget)


def detect_orb_points(gray, budget=220):
    detector = cv.ORB_create(nfeatures=budget, fastThreshold=12)
    return keypoints_to_points(detector.detect(gray, None), budget)


def detect_sift_points(gray, budget=220):
    if not hasattr(cv, "SIFT_create"):
        return np.empty((0, 2), np.float32)
    detector = cv.SIFT_create(nfeatures=budget)
    return keypoints_to_points(detector.detect(gray, None), budget)


detectors = {
    "Shi–Tomasi": detect_shi,
    "FAST": detect_fast,
    "ORB": detect_orb_points,
}
if hasattr(cv, "SIFT_create"):
    detectors["SIFT"] = detect_sift_points

detector_rows = []
detector_points = {}
for name, detector in detectors.items():
    start = time.perf_counter()
    ref_points = detector(gray1, 220)
    target_points = detector(gray2, 220)
    elapsed_ms = (time.perf_counter() - start) * 1000
    result = repeatability(ref_points, target_points, H_gt, img2.shape, epsilon=5)
    detector_points[name] = (ref_points, target_points)
    detector_rows.append({
        "detector": name,
        "N_ref": len(ref_points),
        "N_target": len(target_points),
        "visibili": result["visible"],
        "ripetuti": result["repeated"],
        "rho@5": result["rho"],
        "errore_mediano_px": np.median(result["errors"]) if len(result["errors"]) else np.nan,
        "copertura_ref": grid_coverage(ref_points, img1.shape),
        "tempo_2_immagini_ms": elapsed_ms,
    })

detector_df = pd.DataFrame(detector_rows)
display(detector_df.round(3))

fig, axes = plt.subplots(1, len(detector_points), figsize=(4.5 * len(detector_points), 4.2))
if len(detector_points) == 1: axes = [axes]
for ax, (name, (points, _)) in zip(axes, detector_points.items()):
    canvas = img1.copy()
    for x, y in points:
        cv.circle(canvas, (round(x), round(y)), 4, CV_CORAL, 2, cv.LINE_AA)
    ax.imshow(rgb(canvas)); ax.set_title(name); ax.axis("off")
plt.tight_layout()


**Risposta dello studente.** Quale detector scegliereste per tracking real-time e quale per forti cambiamenti di scala? Motivare usando almeno due colonne.

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

## Lab 5 — Keypoint e descrittore ORB

ORB combina FAST, piramide, orientazione e rBRIEF. Un `KeyPoint` non è soltanto una coppia $(x, y)$: contiene scala, angolo, risposta e ottava. Il descrittore è binario e richiede distanza di Hamming.

In [ ]:
orb = cv.ORB_create(nfeatures=800, fastThreshold=12)
kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)
assert des1 is not None and des2 is not None, "Descrittori mancanti"

print("Keypoint:", len(kp1), len(kp2))
print("Descrittori:", des1.shape, des1.dtype, "|", des2.shape, des2.dtype)
print("Esempio KeyPoint:", {
    "pt": tuple(round(v, 1) for v in kp1[0].pt),
    "size": round(kp1[0].size, 1),
    "angle": round(kp1[0].angle, 1),
    "response": round(kp1[0].response, 4),
    "octave": kp1[0].octave,
})

rich1 = cv.drawKeypoints(img1, kp1, None, color=CV_CYAN,
                         flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
rich2 = cv.drawKeypoints(img2, kp2, None, color=CV_CORAL,
                         flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, image, title in zip(axes, [rich1, rich2], ["ORB — riferimento", "ORB — trasformata"]):
    ax.imshow(rgb(image)); ax.set_title(title); ax.axis("off")
plt.tight_layout()

# Hamming = numero di bit diversi: XOR seguito da popcount.
manual_hamming = int(np.unpackbits(np.bitwise_xor(des1[0], des2[0])).sum())
print("Hamming manuale tra i primi due descrittori:", manual_hamming)


## Lab 6 — Matching: nearest neighbor, ratio e verità geometrica

Il nearest neighbor propone sempre un candidato. Il ratio test accetta il migliore soltanto se è nettamente più vicino del secondo. Poiché $H_{gt}$ è noto, possiamo misurare la precisione geometrica dei match accettati senza usare RANSAC come giudice di sé stesso.

In [ ]:
matcher = cv.BFMatcher(cv.NORM_HAMMING, crossCheck=False)
pairs = [pair for pair in matcher.knnMatch(des1, des2, k=2) if len(pair) == 2]
raw_matches = [pair[0] for pair in pairs]


def match_errors(matches, keypoints1, keypoints2, H):
    if not matches:
        return np.array([], dtype=float)
    source = np.float64([keypoints1[m.queryIdx].pt for m in matches])
    target = np.float64([keypoints2[m.trainIdx].pt for m in matches])
    predicted = project_points(source, H)
    return np.linalg.norm(predicted - target, axis=1)


def estimate_homography(matches, keypoints1, keypoints2, threshold=4.0):
    if len(matches) < 4:
        return None, np.zeros(len(matches), dtype=bool)
    source = np.float32([keypoints1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    target = np.float32([keypoints2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    cv.setRNGSeed(SEED)
    H, mask = cv.findHomography(source, target, cv.RANSAC, threshold)
    if H is None or mask is None:
        return None, np.zeros(len(matches), dtype=bool)
    return H, mask.ravel().astype(bool)


def refit_homography(matches, keypoints1, keypoints2, inlier_mask, initial_H=None):
    # Ristima lineare su tutti gli inlier per valutare il modello finale.
    inlier_matches = [match for match, keep in zip(matches, inlier_mask) if keep]
    if len(inlier_matches) < 4:
        return initial_H
    source = np.float32([keypoints1[m.queryIdx].pt for m in inlier_matches]).reshape(-1, 1, 2)
    target = np.float32([keypoints2[m.trainIdx].pt for m in inlier_matches]).reshape(-1, 1, 2)
    H_refit, _ = cv.findHomography(source, target, method=0)
    return initial_H if H_refit is None else H_refit


def homography_grid_error(H_est, H_true, shape, rows=5, cols=7):
    if H_est is None:
        return np.nan
    height, width = shape[:2]
    xs = np.linspace(0.08 * width, 0.92 * width, cols)
    ys = np.linspace(0.08 * height, 0.92 * height, rows)
    grid = np.array([(x, y) for y in ys for x in xs], dtype=float)
    true_projection = project_points(grid, H_true)
    valid = ((true_projection[:, 0] >= 0) & (true_projection[:, 0] < width) &
             (true_projection[:, 1] >= 0) & (true_projection[:, 1] < height))
    if not np.any(valid):
        return np.nan
    estimated_projection = project_points(grid[valid], H_est)
    return float(np.median(np.linalg.norm(estimated_projection - true_projection[valid], axis=1)))


ratio_threshold = 0.65
ratio_matches = [best for best, second in pairs
                 if best.distance < ratio_threshold * second.distance]
gt_errors = match_errors(ratio_matches, kp1, kp2, H_gt)
print("Raw nearest-neighbor:", len(raw_matches))
print(f"Ratio matches (τ={ratio_threshold:.2f}):", len(ratio_matches))
print("Match geometricamente corretti @4 px:", int(np.sum(gt_errors <= 4.0)))
print("Precisione geometrica @4 px:", f"{np.mean(gt_errors <= 4.0):.3f}")

raw_draw = cv.drawMatches(img1, kp1, img2, kp2,
                          sorted(raw_matches, key=lambda m: m.distance)[:100], None,
                          matchColor=CV_CORAL,
                          flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
ratio_draw = cv.drawMatches(img1, kp1, img2, kp2, ratio_matches[:100], None,
                            matchColor=CV_CYAN,
                            flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
axes[0].imshow(rgb(raw_draw)); axes[0].set_title("Raw nearest-neighbor — primi 100")
axes[1].imshow(rgb(ratio_draw)); axes[1].set_title(f"Ratio test τ={ratio_threshold:.2f} — primi 100")
for ax in axes: ax.axis("off")
plt.tight_layout()


### Esperimento 6A — Ratio sweep

Per ogni soglia riportiamo sia la correttezza rispetto a $H_{gt}$ sia il consenso trovato da RANSAC. Le due quantità non sono sinonimi.

In [ ]:
ratio_rows = []
for threshold in [0.55, 0.65, 0.75, 0.85, 0.95]:
    selected = [best for best, second in pairs if best.distance < threshold * second.distance]
    errors_gt = match_errors(selected, kp1, kp2, H_gt)
    H_tmp, inlier_mask = estimate_homography(selected, kp1, kp2, threshold=4.0)
    H_final = refit_homography(selected, kp1, kp2, inlier_mask, H_tmp)
    n_inliers = int(inlier_mask.sum())
    ratio_rows.append({
        "tau": threshold,
        "match": len(selected),
        "corretti_gt@4": int(np.sum(errors_gt <= 4.0)),
        "precisione_gt@4": np.mean(errors_gt <= 4.0) if len(errors_gt) else np.nan,
        "inlier_RANSAC": n_inliers,
        "inlier_ratio": n_inliers / len(selected) if selected else np.nan,
        "errore_griglia_H_px": homography_grid_error(H_final, H_gt, img1.shape),
    })

ratio_df = pd.DataFrame(ratio_rows)
display(ratio_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(ratio_df["tau"], ratio_df["match"], "o-", label="match")
axes[0].plot(ratio_df["tau"], ratio_df["corretti_gt@4"], "o-", label="corretti GT")
axes[0].set(xlabel="soglia ratio τ", ylabel="conteggio", title="Disponibilità di evidenze")
axes[0].legend(); axes[0].grid(alpha=0.25)
axes[1].plot(ratio_df["tau"], ratio_df["precisione_gt@4"], "o-", label="precisione GT")
axes[1].plot(ratio_df["tau"], ratio_df["inlier_ratio"], "o-", label="inlier ratio")
axes[1].set(xlabel="soglia ratio τ", ylabel="frazione", ylim=(0, 1.03),
            title="Precisione GT e consenso al modello")
axes[1].legend(); axes[1].grid(alpha=0.25)
plt.tight_layout()


**Risposta dello studente.** Quale soglia usereste per inizializzare l'omografia? Spiegare il compromesso fra numero di match, precisione e copertura.

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

## Lab 7 — RANSAC, errore e controllo fuori dai match

Fissiamo il set di match e variamo soltanto la soglia di riproiezione di RANSAC. Misuriamo:

- numero e rapporto di inlier;
- errore di riproiezione mediano e percentile 90 sugli inlier;
- errore della trasformazione su una griglia di punti che non coincide con i match.

In [ ]:
def reprojection_errors(matches, keypoints1, keypoints2, H):
    return match_errors(matches, keypoints1, keypoints2, H) if H is not None else np.array([])


ransac_rows = []
for geometric_threshold in [1.0, 2.0, 3.0, 4.0, 6.0, 8.0]:
    H_tmp, mask_tmp = estimate_homography(ratio_matches, kp1, kp2, geometric_threshold)
    H_final = refit_homography(ratio_matches, kp1, kp2, mask_tmp, H_tmp)
    all_errors = reprojection_errors(ratio_matches, kp1, kp2, H_final)
    inlier_errors = all_errors[mask_tmp] if len(all_errors) else np.array([])
    ransac_rows.append({
        "soglia_px": geometric_threshold,
        "inlier": int(mask_tmp.sum()),
        "inlier_ratio": mask_tmp.mean() if len(mask_tmp) else np.nan,
        "mediana_px": np.median(inlier_errors) if len(inlier_errors) else np.nan,
        "p90_px": np.percentile(inlier_errors, 90) if len(inlier_errors) else np.nan,
        "errore_griglia_H_px": homography_grid_error(H_final, H_gt, img1.shape),
    })

ransac_df = pd.DataFrame(ransac_rows)
display(ransac_df.round(3))


In [ ]:
RANSAC_THRESHOLD = 4.0
H_est, inlier_mask = estimate_homography(ratio_matches, kp1, kp2, RANSAC_THRESHOLD)
H_est = refit_homography(ratio_matches, kp1, kp2, inlier_mask, H_est)
inliers = [match for match, keep in zip(ratio_matches, inlier_mask) if keep]
outliers = [match for match, keep in zip(ratio_matches, inlier_mask) if not keep]
errors = reprojection_errors(ratio_matches, kp1, kp2, H_est)
inlier_errors = errors[inlier_mask] if len(errors) else np.array([])

print("Match dopo ratio:", len(ratio_matches))
print("Inlier:", len(inliers), f"({inlier_mask.mean():.3f})" if len(inlier_mask) else "(n.d.)")
print("Errore inlier mediano / p90:",
      f"{np.median(inlier_errors):.2f} / {np.percentile(inlier_errors, 90):.2f} px"
      if len(inlier_errors) else "n.d.: nessun inlier con errore valutabile")
print("Errore mediano di H sulla griglia:",
      f"{homography_grid_error(H_est, H_gt, img1.shape):.2f} px")

inlier_draw = cv.drawMatches(img1, kp1, img2, kp2, inliers[:120], None,
                             matchColor=CV_CYAN,
                             flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
outlier_draw = cv.drawMatches(img1, kp1, img2, kp2, outliers[:120], None,
                              matchColor=CV_CORAL,
                              flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

localized = img2.copy()
height, width = img1.shape[:2]
contour = np.float32([[0, 0], [width - 1, 0],
                      [width - 1, height - 1], [0, height - 1]]).reshape(-1, 1, 2)
if H_est is not None:
    projected_contour = cv.perspectiveTransform(contour, H_est)
    cv.polylines(localized, [np.int32(projected_contour)], True, CV_CORAL, 5, cv.LINE_AA)
else:
    print("Omografia non stimabile: il contorno non viene disegnato.")

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
axes[0].imshow(rgb(inlier_draw)); axes[0].set_title("Consenso RANSAC — inlier")
axes[1].imshow(rgb(outlier_draw)); axes[1].set_title("Match esclusi dal consenso RANSAC")
axes[2].imshow(rgb(localized)); axes[2].set_title("Contorno proiettato: controllo fuori dai match")
for ax in axes: ax.axis("off")
plt.tight_layout()


**Risposta dello studente.** Perché una soglia RANSAC più grande aumenta spesso l'inlier ratio ma non garantisce una trasformazione più accurata?

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

## Lab 8 — Failure clinic automatizzata

Costruiamo tre casi in cui falliscono assunzioni diverse:

1. **texture ripetitiva:** molti intorni simili;
2. **blur + scala estrema:** la struttura esce dal supporto del detector/descrittore;
3. **occlusione + riflesso sintetico:** alcune vere corrispondenze non esistono più.

La tabella serve a individuare il primo stadio che degrada. Le immagini servono a verificare che la spiegazione sia plausibile.

Nello scenario con occlusione il denominatore della ripetibilità conserva i centri geometricamente interni, anche quando sono coperti: il calo non misura soltanto il detector.

In [ ]:
def make_repetitive_board(width=960, height=640, tile=80):
    image = np.full((height, width, 3), (236, 232, 222), np.uint8)
    for y in range(20, height - tile, tile):
        for x in range(20, width - tile, tile):
            cv.rectangle(image, (x, y), (x + 54, y + 54), CV_INK, 4)
            cv.circle(image, (x + 27, y + 27), 14, CV_CYAN, 4, cv.LINE_AA)
            cv.line(image, (x + 8, y + 8), (x + 46, y + 46), CV_CORAL, 4, cv.LINE_AA)
    return image


repetitive_ref = make_repetitive_board()
repetitive_target, H_repetitive = warp_test(
    repetitive_ref, angle=0, scale=1.0, tx=73, ty=17,
    gain=0.92, bias=8, blur=0.5)

blur_ref = img1.copy()
blur_target, H_blur = warp_test(
    blur_ref, angle=18, scale=0.52, tx=120, ty=45,
    gain=0.82, bias=25, blur=3.2)

occlusion_ref = img1.copy()
occlusion_target, H_occlusion = warp_test(
    occlusion_ref, angle=20, scale=0.90, tx=40, ty=15,
    gain=0.85, bias=22, blur=0.9)
cv.rectangle(occlusion_target, (430, 120), (760, 380), (245, 245, 250), -1)
cv.ellipse(occlusion_target, (720, 465), (145, 58), -18, 0, 360, (255, 255, 255), -1, cv.LINE_AA)

scenarios = {
    "texture ripetitiva": (repetitive_ref, repetitive_target, H_repetitive),
    "blur + scala": (blur_ref, blur_target, H_blur),
    "occlusione + riflesso": (occlusion_ref, occlusion_target, H_occlusion),
}


def analyze_scenario(reference, target, H_true, ratio=0.75):
    gray_ref, gray_target = gray_of(reference), gray_of(target)
    shi_ref, shi_target = detect_shi(gray_ref), detect_shi(gray_target)
    rep = repeatability(shi_ref, shi_target, H_true, target.shape, epsilon=5)

    local_orb = cv.ORB_create(nfeatures=800, fastThreshold=12)
    key_ref, desc_ref = local_orb.detectAndCompute(gray_ref, None)
    key_target, desc_target = local_orb.detectAndCompute(gray_target, None)
    if desc_ref is None or desc_target is None:
        return {"rho_detector": rep["rho"], "kp_ref": len(key_ref), "kp_target": len(key_target),
                "ratio_match": 0, "precisione_gt": np.nan, "inlier_ratio": np.nan,
                "errore_H_griglia": np.nan}, None

    local_pairs = [pair for pair in cv.BFMatcher(cv.NORM_HAMMING).knnMatch(desc_ref, desc_target, k=2)
                   if len(pair) == 2]
    selected = [best for best, second in local_pairs if best.distance < ratio * second.distance]
    gt_err = match_errors(selected, key_ref, key_target, H_true)
    H_local, mask_local = estimate_homography(selected, key_ref, key_target, threshold=4.0)
    H_local = refit_homography(selected, key_ref, key_target, mask_local, H_local)
    metrics = {
        "rho_detector": rep["rho"],
        "kp_ref": len(key_ref),
        "kp_target": len(key_target),
        "ratio_match": len(selected),
        "precisione_gt": np.mean(gt_err <= 4.0) if len(gt_err) else np.nan,
        "inlier_ratio": mask_local.mean() if len(mask_local) else np.nan,
        "errore_H_griglia": homography_grid_error(H_local, H_true, reference.shape),
    }
    inlier_selected = [m for m, keep in zip(selected, mask_local) if keep]
    drawing = cv.drawMatches(reference, key_ref, target, key_target, inlier_selected[:100], None,
                             matchColor=CV_CYAN,
                             flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    return metrics, drawing


failure_rows, failure_drawings = [], {}
for name, (reference, target, H_true) in scenarios.items():
    metrics, drawing = analyze_scenario(reference, target, H_true)
    failure_rows.append({"scenario": name, **metrics})
    failure_drawings[name] = drawing

failure_df = pd.DataFrame(failure_rows)
display(failure_df.round(3))

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
for row, (name, (reference, target, _)) in enumerate(scenarios.items()):
    axes[row, 0].imshow(rgb(np.hstack([reference, target])))
    axes[row, 0].set_title(f"{name}: coppia")
    if failure_drawings[name] is not None:
        axes[row, 1].imshow(rgb(failure_drawings[name]))
    axes[row, 1].set_title("inlier RANSAC dopo ratio test")
    axes[row, 0].axis("off"); axes[row, 1].axis("off")
plt.tight_layout()


**Risposta dello studente.** Per ciascuno scenario indicare il primo stadio che fallisce e una misura che lo dimostra.

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.

## Lab 9 — Esperimento su immagini reali

Acquisire due fotografie di un oggetto prevalentemente piano e ricco di dettagli, per esempio la copertina di un libro o un poster:

1. immagine A quasi frontale;
2. immagine B con rotazione, cambio di scala o punto di vista;
3. opzionalmente una terza immagine con blur, riflesso o occlusione.

Evitare volti e dati personali. Salvare i file accanto al notebook oppure caricarli nel pannello file di Colab. Inserire i percorsi qui sotto. Non conoscendo $H_{gt}$, potremo misurare consenso ed errore di riproiezione, ma non la precisione geometrica assoluta dei match.

In [ ]:
REAL_IMAGE_1 = ""  # esempio: "book_01.jpg"
REAL_IMAGE_2 = ""  # esempio: "book_02.jpg"


def analyze_real_pair(path1, path2, ratio=0.75, ransac_threshold=4.0):
    image_a, image_b = cv.imread(str(path1)), cv.imread(str(path2))
    if image_a is None or image_b is None:
        raise FileNotFoundError("Controllare i percorsi delle due immagini")
    detector = cv.ORB_create(nfeatures=1500, fastThreshold=12)
    key_a, desc_a = detector.detectAndCompute(gray_of(image_a), None)
    key_b, desc_b = detector.detectAndCompute(gray_of(image_b), None)
    if desc_a is None or desc_b is None:
        raise RuntimeError("Nessun descrittore: scegliere immagini più strutturate")
    candidate_pairs = [p for p in cv.BFMatcher(cv.NORM_HAMMING).knnMatch(desc_a, desc_b, k=2)
                       if len(p) == 2]
    selected = [m for m, n in candidate_pairs if m.distance < ratio * n.distance]
    H, mask = estimate_homography(selected, key_a, key_b, ransac_threshold)
    inliers_local = [m for m, keep in zip(selected, mask) if keep]
    errors_local = reprojection_errors(selected, key_a, key_b, H)
    inlier_errors_local = errors_local[mask] if len(errors_local) else np.array([])
    drawing = cv.drawMatches(image_a, key_a, image_b, key_b, inliers_local[:150], None,
                             matchColor=CV_CYAN,
                             flags=cv.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    metrics = {
        "keypoint_A": len(key_a), "keypoint_B": len(key_b),
        "ratio_match": len(selected), "inlier": len(inliers_local),
        "inlier_ratio": mask.mean() if len(mask) else np.nan,
        "errore_mediano_px": np.median(inlier_errors_local) if len(inlier_errors_local) else np.nan,
        "p90_px": np.percentile(inlier_errors_local, 90) if len(inlier_errors_local) else np.nan,
    }
    return metrics, drawing


if REAL_IMAGE_1 and REAL_IMAGE_2:
    real_metrics, real_drawing = analyze_real_pair(REAL_IMAGE_1, REAL_IMAGE_2)
    display(pd.DataFrame([real_metrics]).round(3))
    plt.figure(figsize=(16, 7)); plt.imshow(rgb(real_drawing)); plt.axis("off");
    plt.title("Inlier sulla coppia reale")
else:
    print("Esperimento reale non ancora eseguito: impostare REAL_IMAGE_1 e REAL_IMAGE_2.")


## Discussione conclusiva e rapporto scientifico

Consegnare:

1. curva $\rho(\varepsilon)$ e heatmap rotazione/scala;
2. tabella dei detector a parità di budget;
3. ratio sweep con precisione e inlier ratio;
4. raw match, inlier, outlier e contorno proiettato;
5. esperimento su immagini reali;
6. diagnosi finale nel formato seguente.

> **Trasformazione:** ...  
> **Primo stadio che fallisce:** ...  
> **Evidenza visuale:** ...  
> **Misura numerica:** ...  
> **Modifica proposta e previsione verificabile:** ...

### Rubrica rapida — 10 punti

| Criterio | Punti |
|---|---:|
| protocollo controllato e parametri dichiarati | 2 |
| correttezza di metriche e figure | 2 |
| confronto equo e interpretazione | 2 |
| error analysis e distinzione detector/matcher/modello | 3 |
| riproducibilità e chiarezza | 1 |

**Fonti:** A. Distante e C. Distante, capitolo *Rilevatori e descrittori di punti di interesse*; R. Szeliski, *Computer Vision: Algorithms and Applications*, 2a ed., cap. 7; Lowe, IJCV 2004; documentazione ufficiale OpenCV 4.x.

**Risposta dello studente.** Completare la diagnosi finale usando il caso reale o il caso più istruttivo della failure clinic.

> Scrivere qui una risposta di 2–5 righe, citando almeno una misura e una figura.